# AEVUM Stage 1 -- Colab training

Epoch-based training (tqdm progress bar, one logged summary line per epoch). Data and checkpoints are written directly to Google Drive as training runs, so a disconnected/idle Colab session only loses progress since the last epoch, not the whole run -- no manual archive/download step needed.

If the runtime disconnects mid-run: just re-run all cells, then add `--resume /content/drive/MyDrive/aevum/outputs/stage1_final.pt` (or a specific `stage1_epochN.pt`) to the training command in the last cell before running it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/karl4th/aevum.git /content/aevum 2>/dev/null || (cd /content/aevum && git pull)

In [ ]:
%cd /content/aevum
!pip install -q uv
!uv sync

In [ ]:
!mkdir -p /content/drive/MyDrive/aevum/data /content/drive/MyDrive/aevum/outputs

`--librispeech-url train-clean-100` (100h, ~6.3GB download -- first run will take a while to fetch, cached on Drive after that since `--data-root` points there). ~2812 steps/epoch at batch 64 x 2s segments, so `--epochs 18` is the 50k-step budget (18*2812 = 50616).

Recompute `--epochs` if you change `--librispeech-url` or `--batch-size` -- steps/epoch = dataset_hours*3600 / (batch_size*segment_seconds), also printed as `steps_per_epoch` in the run's `config` block in `train_log.json` once it starts.

In [ ]:
!uv run python scripts/train_stage1.py \
  --data-root /content/drive/MyDrive/aevum/data \
  --checkpoint-dir /content/drive/MyDrive/aevum/outputs \
  --librispeech-url train-clean-100 \
  --lr 3e-4 \
  --epochs 18